In [1]:

#!pip uninstall -y numpy pandas
!pip install numpy==1.26.4 pandas==2.1.4
!pip install -q sae-lens transformer-lens transformers datasets huggingface_hub plotly

print("\n✓ Installation complete!")
print("\n⚠️  IMPORTANT: Please restart the runtime now:")
print("   Runtime > Restart runtime")
print("\nThen skip this cell and run from 'Imports' section.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 147.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.1.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 w

ERROR: Operation cancelled by user
^C

✓ Installation complete!

⚠️  IMPORTANT: Please restart the runtime now:
   Runtime > Restart runtime

Then skip this cell and run from 'Imports' section.


In [1]:
!pip install sae_lens

  Using cached sae_lens-6.27.3-py3-none-any.whl.metadata (5.4 kB)
  Using cached babe-0.0.7-py3-none-any.whl.metadata (10 kB)
  Using cached plotly_express-0.4.1-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached transformer_lens-2.16.1-py3-none-any.whl.metadata (12 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached beartype-0.14.1-py3-none-any.whl.metadata (28 kB)
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.5/220.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.7/273.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [2]:
# Verify package versions first
import sys
print("Python version:", sys.version)

try:
    import numpy as np
    print(f"NumPy version: {np.__version__}")

    import pandas as pd
    print(f"Pandas version: {pd.__version__}")

    import torch
    print(f"PyTorch version: {torch.__version__}")

except Exception as e:
    print(f"\n❌ Import error: {e}")
    print("\nPlease run the installation cell and restart runtime.")
    raise

# Continue with other imports
from typing import List, Dict, Tuple
from dataclasses import dataclass
import json
from collections import defaultdict

from sae_lens import SAE, HookedSAETransformer
from transformer_lens import HookedTransformer
from datasets import load_dataset
import plotly.express as px
import plotly.graph_objects as go
from tqdm.auto import tqdm

# Disable gradients for efficiency
torch.set_grad_enabled(False)

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n✓ All imports successful!")

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
NumPy version: 1.26.4
Pandas version: 2.1.4
PyTorch version: 2.9.0+cu126

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.17 GB

✓ All imports successful!


## 2. Configuration

In [4]:
# Model and SAE configuration
MODEL_NAME = "google/gemma-2-9b-it"  # 9B instruction-tuned model
SAE_RELEASE = "gemma-scope-9b-it-res-canonical"
SAE_ID = "layer_20/width_131k/canonical"  # Layer 20, 131K width SAE

# Slang features to analyze (from your research on Layer 20)
SLANG_FEATURES = {
    "universal": [35440, 33236, 93521],
    "intensity": [51811],
    "literalness": [90871]
}

# TokenChange method parameters (from paper Section 4)
K_RANDOM_PROMPTS = 32  # Number of random prompts
PROMPT_LENGTH = 32  # Token length per prompt
TOP_K_TOKENS = 20  # Top tokens to analyze for change
AMPLIFICATION_VALUES = [0.5, 1.0, 5.0,10.0]  # Different amplification levels to test

## 3. Load Model and SAE

In [3]:
from transformers import AutoTokenizer
from huggingface_hub import notebook_login
import numpy as np
import torch

# Login to HuggingFace
notebook_login()

In [ ]:
!pip install transformer_lens

  Using cached transformer_lens-2.16.1-py3-none-any.whl.metadata (12 kB)
  Using cached beartype-0.14.1-py3-none-any.whl.metadata (28 kB)
  Using cached better_abc-0.0.3-py3-none-any.whl.metadata (1.4 kB)
  Using cached fancy_einsum-0.0.3-py3-none-any.whl.metadata (1.2 kB)
  Using cached jaxtyping-0.3.4-py3-none-any.whl.metadata (7.8 kB)
  Using cached transformers-stream-generator-0.0.5.tar.gz (13 kB)
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 5.9 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=fc914670c682625f8e7f5c1cf114d594b73f3f5998aea16bfe6ab56f10cc053a
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfull

In [5]:
print("Loading Gemma 9B IT model with TransformerLens...")
print("This will take a few minutes...")
import transformer_lens
# Load the model using TransformerLens for activation extraction
tl_model = transformer_lens.HookedTransformer.from_pretrained(
    "google/gemma-2-9b-it",
    device="cuda",
    dtype=torch.bfloat16,
)

print("Model loaded successfully!")
print(f"Model has {tl_model.cfg.n_layers} layers")
print(f"Model dimension: {tl_model.cfg.d_model}")

print("Loading SAE from Gemma Scope...")
print("Repository: google/gemma-scope-9b-it-res")
print("Layer: 20 (LATE LAYER - closer to output), Width: 131k")

# Load the Layer 20 SAE
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-9b-it-res-canonical",
    sae_id="layer_20/width_131k/canonical",
    device="cuda"
)

print(f"\nSAE loaded successfully!")
print(f"SAE width (number of features): {sae.cfg.d_sae}")
print(f"SAE input dimension: {sae.cfg.d_in}")
print(f"Average L0 (sparsity): {sparsity}")
print(f"Hook point: blocks.20.hook_resid_post")

Loading Gemma 9B IT model with TransformerLens...
This will take a few minutes...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model google/gemma-2-9b-it into HookedTransformer
Model loaded successfully!
Model has 42 layers
Model dimension: 3584
Loading SAE from Gemma Scope...
Repository: google/gemma-scope-9b-it-res
Layer: 20 (LATE LAYER - closer to output), Width: 131k


layer_20/width_131k/average_l0_81/params(…):   0%|          | 0.00/3.76G [00:00<?, ?B/s]


SAE loaded successfully!
SAE width (number of features): 131072
SAE input dimension: 3584
Average L0 (sparsity): None
Hook point: blocks.20.hook_resid_post


/tmp/ipython-input-4120432844.py:20: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


## 4. Data Preparation

We'll use random prompts from The Pile dataset, as described in the paper.

In [6]:
def get_slang_dataset_prompts(n_prompts_per_dataset: int = 10) -> List[str]:
    """
    Load prompts from slang research datasets.

    Args:
        n_prompts_per_dataset: Number of prompts to sample from each dataset

    Returns:
        List of text prompts from slang datasets
    """
    print(f"Loading prompts from slang research datasets...")

    all_prompts = []

    # Define the datasets and their sentence column names
    datasets_info = [
        ("hebrew_slang_classification_results.csv", ["sentence", "Sentence", "SENTENCE"]),
        ("hebrew_slang_dataset.csv", ["sentence", "Sentence", "SENTENCE"]),
        ("russian_literal_negatives_dataset.csv", ["text", "Sentence", "SENTENCE"]),
        ("russian_slang_positive_results.csv", ["text", "Sentence", "SENTENCE"]),
        ("unified_english_dataset.csv", ["sentence", "Sentence", "SENTENCE"]),
        ("negatives_with_term.csv", ["sentence", "Sentence", "SENTENCE"])
    ]

    for dataset_file, possible_columns in datasets_info:
        try:
            print(f"\n  Loading from {dataset_file}...")

            # Try to load the dataset
            df = pd.read_csv(dataset_file)

            # Find which column name exists
            sentence_col = None
            for col in possible_columns:
                if col in df.columns:
                    sentence_col = col
                    break

            if sentence_col is None:
                print(f"    ⚠️  No sentence column found. Available columns: {df.columns.tolist()}")
                continue

            # Sample sentences
            sentences = df[sentence_col].dropna().tolist()

            # Take up to n_prompts_per_dataset
            sampled = sentences[:n_prompts_per_dataset] if len(sentences) >= n_prompts_per_dataset else sentences

            all_prompts.extend(sampled)
            print(f"    ✓ Loaded {len(sampled)} sentences")

        except FileNotFoundError:
            print(f"    ❌ File not found: {dataset_file}")
        except Exception as e:
            print(f"    ❌ Error loading {dataset_file}: {e}")

    print(f"\n✓ Total sentences loaded from datasets: {len(all_prompts)}")
    return all_prompts




# Load prompts from slang datasets
dataset_prompts = get_slang_dataset_prompts(n_prompts_per_dataset=20)

all_prompts = dataset_prompts

print(f"\n📊 Total prompts for analysis: {len(all_prompts)}")
print(f"   - From slang datasets: {len(dataset_prompts)}")

# Show some examples
if all_prompts:
    print(f"\n🎯 Example prompts:")
    for i, prompt in enumerate(all_prompts[:5], 1):
        preview = prompt[:80] + "..." if len(prompt) > 80 else prompt
        print(f"   {i}. \"{preview}\"")
    if len(all_prompts) > 5:
        print(f"   ... and {len(all_prompts) - 5} more")
else:
    print("\n⚠️  WARNING: No prompts loaded!")

Loading prompts from slang research datasets...

  Loading from hebrew_slang_classification_results.csv...
    ✓ Loaded 20 sentences

  Loading from hebrew_slang_dataset.csv...
    ✓ Loaded 20 sentences

  Loading from russian_literal_negatives_dataset.csv...
    ✓ Loaded 20 sentences

  Loading from russian_slang_positive_results.csv...
    ✓ Loaded 20 sentences

  Loading from unified_english_dataset.csv...
    ✓ Loaded 20 sentences

  Loading from negatives_with_term.csv...
    ✓ Loaded 20 sentences

✓ Total sentences loaded from datasets: 120

📊 Total prompts for analysis: 120
   - From slang datasets: 120

🎯 Example prompts:
   1. "אתה לא נשיא במקרה ? ? ? ? כן זה שהתפקיד שלו זה לבזבז לנו את כספי הציבור על הטיסו..."
   2. "אש אתה ! ! !"
   3. "שמעו . . היהודים נהיו כזה עם דפוק.העם הגזעני בעולם.חבורת קיצונים משועממים"
   4. "מה קרה ? מאז שנהיית נשיא התחלת להתחנף רקקק לערבים ! תתבייש לך ! אחר כך שהיא תחטו..."
   5. "למה אתה פותח את הפה שלך בלי שביקשו ממך ? למה אתה מוציא את עצמך דפוק 

## 6. VocabProj Method Implementation

From the paper (Section 4):
> "We propose to view the feature f as an update to the model's output distribution. To interpret f's contribution, we compute the feature vector F⁻¹(f) = vf ∈ ℝᵈ and project it to the vocabulary space to obtain a vector of logits w ∈ ℝ|V| such that: w = WU LayerNorm(vf) where V is M's vocabulary, LayerNorm is the final layer norm, and WU ∈ ℝ|V|×d is the model's unembedding matrix."

This method is:
- **Extremely efficient**: Just a single matrix multiplication
- **Correlative**: Shows direct relationship between feature and vocabulary
- **Complementary to TokenChange**: VocabProj is correlative, TokenChange is causal

In [10]:
@dataclass
class VocabProjResult:
    """Results from VocabProj analysis"""
    feature_id: int
    feature_category: str
    top_promoted_tokens: List[Tuple[str, float]]  # (token, logit_weight)
    top_suppressed_tokens: List[Tuple[str, float]]
    mean_absolute_weight: float
    max_weight: float


def vocabproj_analysis(
    feature_id: int,
    feature_category: str,
    top_k: int = 50
) -> VocabProjResult:
    """
    Analyze a feature using the VocabProj method.

    Args:
        feature_id: SAE feature index to analyze
        feature_category: Category label (e.g., 'universal', 'intensity')
        top_k: Number of top tokens to return

    Returns:
        VocabProjResult with analysis results
    """
    print(f"\nAnalyzing feature {feature_id} ({feature_category}) with VocabProj...")

    # 1. Get the feature vector (decoder direction)
    # The SAE decoder matrix W_dec maps from feature space to model activation space
    feature_vector = sae.W_dec[feature_id, :]  # Shape: [d_model]

    # 2. Apply LayerNorm (model's final layer norm)
    # Get the layer norm from the model
    ln = tl_model.ln_final
    normalized_feature = ln(feature_vector.unsqueeze(0)).squeeze(0)

    # 3. Project to vocabulary space using unembedding matrix
    W_U = tl_model.W_U  # Shape: [d_model, vocab_size]
    vocab_logits = normalized_feature @ W_U  # Shape: [vocab_size]

    # 4. Get top promoted and suppressed tokens
    vocab_logits_np = vocab_logits.float().cpu().numpy()

    # Get indices of top and bottom tokens
    top_indices = np.argsort(vocab_logits_np)[-top_k:][::-1]
    bottom_indices = np.argsort(vocab_logits_np)[:top_k]

    top_promoted = [
        (tl_model.to_string(int(idx)), float(vocab_logits_np[idx]))
        for idx in top_indices
    ]

    top_suppressed = [
        (tl_model.to_string(int(idx)), float(vocab_logits_np[idx]))
        for idx in bottom_indices
    ]

    # 5. Calculate statistics
    mean_abs_weight = float(np.mean(np.abs(vocab_logits_np)))
    max_weight = float(np.max(np.abs(vocab_logits_np)))

    print(f"  Mean absolute weight: {mean_abs_weight:.4f}")
    print(f"  Max weight: {max_weight:.4f}")

    return VocabProjResult(
        feature_id=feature_id,
        feature_category=feature_category,
        top_promoted_tokens=top_promoted,
        top_suppressed_tokens=top_suppressed,
        mean_absolute_weight=mean_abs_weight,
        max_weight=max_weight
    )

## 7. Run VocabProj Analysis

In [11]:
# Store VocabProj results
vocabproj_results = []

print("\n" + "="*80)
print("RUNNING VOCABPROJ ANALYSIS ON SLANG FEATURES")
print("="*80)

for category, feature_ids in SLANG_FEATURES.items():
    print(f"\n{category.upper()} features:")
    for feature_id in feature_ids:
        result = vocabproj_analysis(
            feature_id=feature_id,
            feature_category=category,
            top_k=50  # Get top 50 for cross-lingual analysis
        )
        vocabproj_results.append(result)

print(f"\n\nCompleted VocabProj analysis of {len(vocabproj_results)} features")


RUNNING VOCABPROJ ANALYSIS ON SLANG FEATURES

UNIVERSAL features:

Analyzing feature 35440 (universal) with VocabProj...
  Mean absolute weight: 5.7752
  Max weight: 47.5000

Analyzing feature 33236 (universal) with VocabProj...
  Mean absolute weight: 4.7431
  Max weight: 44.7500

Analyzing feature 93521 (universal) with VocabProj...
  Mean absolute weight: 7.2927
  Max weight: 38.2500

INTENSITY features:

Analyzing feature 51811 (intensity) with VocabProj...
  Mean absolute weight: 5.9565
  Max weight: 45.0000

LITERALNESS features:

Analyzing feature 90871 (literalness) with VocabProj...
  Mean absolute weight: 6.3604
  Max weight: 45.7500


Completed VocabProj analysis of 5 features


## 8. Display VocabProj Results

In [13]:
def display_vocabproj_results(result: VocabProjResult, n_tokens: int = 20):
    """
    Display detailed VocabProj results for a single feature.
    """
    print(f"\n{'='*80}")
    print(f"VocabProj: Feature {result.feature_id} ({result.feature_category})")
    print(f"{'='*80}")

    print(f"\nMean Absolute Weight: {result.mean_absolute_weight:.4f}")
    print(f"Max Weight: {result.max_weight:.4f}")

    print(f"\n{'─'*80}")
    print(f"TOP {n_tokens} PROMOTED TOKENS:")
    print(f"{'─'*80}")
    for i, (token, weight) in enumerate(result.top_promoted_tokens[:n_tokens], 1):
        print(f"{i:2d}. {token:25s} {weight:8.4f}")

    print(f"\n{'─'*80}")
    print(f"TOP {n_tokens} SUPPRESSED TOKENS:")
    print(f"{'─'*80}")
    for i, (token, weight) in enumerate(result.top_suppressed_tokens[:n_tokens], 1):
        print(f"{i:2d}. {token:25s} {weight:8.4f}")


# Display all VocabProj results
print("\n" + "#"*80)
print("# VOCABPROJ DETAILED RESULTS")
print("#"*80)

for result in vocabproj_results:
    display_vocabproj_results(result)


################################################################################
# VOCABPROJ DETAILED RESULTS
################################################################################

VocabProj: Feature 35440 (universal)

Mean Absolute Weight: 5.7752
Max Weight: 47.5000

────────────────────────────────────────────────────────────────────────────────
TOP 20 PROMOTED TOKENS:
────────────────────────────────────────────────────────────────────────────────
 1.  dudes                     47.5000
 2. hoeddwyd                   45.7500
 3.  guys                      43.7500
 4.  funky                     43.0000
 5.  fellas                    42.5000
 6.  reggae                    41.5000
 7.  dude                      41.2500
 8.  homie                     41.0000
 9.  rappers                   41.0000
10.  badass                    40.5000
11.  streetwear                40.5000
12.  crew                      40.2500
13.  rapper                    39.5000
14.  الرياضيه             

## 9. Cross-Lingual Analysis of Slang Features

### Hypothesis:
If these SAE features truly capture **abstract slang concepts** (not just specific English slang words), they should:
1. Promote slang-like tokens across multiple languages
2. Preserve the ambiguity/informal nature of slang in different linguistic contexts
3. Show similar semantic patterns in non-English tokens

### Method:
We'll analyze the top promoted tokens from VocabProj to identify:
- Non-English tokens (Spanish, French, German, Arabic, Chinese, etc.)
- Whether these tokens represent informal/slang usage in their respective languages
- Semantic alignment across languages

In [14]:
import re
from collections import defaultdict

def detect_language_script(token: str) -> str:
    """
    Detect the script/language family of a token based on Unicode ranges.
    """
    # Remove leading/trailing whitespace and special chars
    token = token.strip()

    if not token:
        return "empty"

    # Check for different Unicode ranges
    has_latin = bool(re.search(r'[a-zA-ZÀ-ÿ]', token))
    has_cyrillic = bool(re.search(r'[\u0400-\u04FF]', token))
    has_arabic = bool(re.search(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]', token))
    has_chinese = bool(re.search(r'[\u4E00-\u9FFF]', token))
    has_japanese = bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF]', token))
    has_korean = bool(re.search(r'[\uAC00-\uD7AF\u1100-\u11FF]', token))
    has_devanagari = bool(re.search(r'[\u0900-\u097F]', token))
    has_thai = bool(re.search(r'[\u0E00-\u0E7F]', token))
    has_hebrew = bool(re.search(r'[\u0590-\u05FF]', token))
    has_greek = bool(re.search(r'[\u0370-\u03FF]', token))

    # Prioritize non-Latin scripts
    if has_chinese:
        return "Chinese"
    elif has_japanese:
        return "Japanese"
    elif has_korean:
        return "Korean"
    elif has_arabic:
        return "Arabic"
    elif has_cyrillic:
        return "Cyrillic/Russian"
    elif has_devanagari:
        return "Devanagari/Hindi"
    elif has_thai:
        return "Thai"
    elif has_hebrew:
        return "Hebrew"
    elif has_greek:
        return "Greek"
    elif has_latin:
        # Try to identify specific Latin-based languages
        # Look for characteristic diacritics and patterns
        if re.search(r'[àâäæçéèêëïîôùûüÿœ]', token.lower()):
            return "French"
        elif re.search(r'[áéíñóúü¿¡]', token.lower()):
            return "Spanish"
        elif re.search(r'[äöüß]', token.lower()):
            return "German"
        elif re.search(r'[àèéìíîòóùú]', token.lower()):
            return "Italian"
        elif re.search(r'[ãáâàçéêíóôõú]', token.lower()):
            return "Portuguese"
        else:
            return "English/Latin"
    else:
        return "Other/Symbol"


def analyze_cross_lingual_tokens(result: VocabProjResult, n_tokens: int = 50):
    """
    Analyze the language distribution in promoted tokens.
    """
    language_counts = defaultdict(list)

    # Analyze top promoted tokens
    for token, weight in result.top_promoted_tokens[:n_tokens]:
        lang = detect_language_script(token)
        language_counts[lang].append((token, weight))

    return language_counts


# Analyze cross-lingual distribution for all features
print("\n" + "="*80)
print("CROSS-LINGUAL ANALYSIS OF PROMOTED TOKENS")
print("="*80)

crosslingual_analysis = {}

for result in vocabproj_results:
    print(f"\n{'─'*80}")
    print(f"Feature {result.feature_id} ({result.feature_category})")
    print(f"{'─'*80}")

    lang_dist = analyze_cross_lingual_tokens(result)
    crosslingual_analysis[result.feature_id] = lang_dist

    # Display language distribution
    print(f"\nLanguage Distribution in Top 50 Tokens:")
    for lang, tokens in sorted(lang_dist.items(), key=lambda x: len(x[1]), reverse=True):
        print(f"  {lang:20s}: {len(tokens):2d} tokens")
        # Show a few examples
        examples = tokens[:3]
        for token, weight in examples:
            print(f"    - {token:20s} ({weight:.4f})")


CROSS-LINGUAL ANALYSIS OF PROMOTED TOKENS

────────────────────────────────────────────────────────────────────────────────
Feature 35440 (universal)
────────────────────────────────────────────────────────────────────────────────

Language Distribution in Top 50 Tokens:
  English/Latin       : 47 tokens
    -  dudes               (47.5000)
    - hoeddwyd             (45.7500)
    -  guys                (43.7500)
  Other/Symbol        :  2 tokens
    -  ***!                (35.7500)
    - 🤘                    (33.5000)
  Arabic              :  1 tokens
    -  الرياضيه            (38.0000)

────────────────────────────────────────────────────────────────────────────────
Feature 33236 (universal)
────────────────────────────────────────────────────────────────────────────────

Language Distribution in Top 50 Tokens:
  English/Latin       : 39 tokens
    - LookAnd              (44.7500)
    - MigrationBuilder     (35.0000)
    - WebVitals            (35.0000)
  Cyrillic/Russian    :  5 t

## 10. Semantic Cross-Lingual Analysis

Now let's analyze if the non-English tokens preserve the slang/informal semantics.

In [15]:
# Common slang indicators across languages
SLANG_INDICATORS = {
    "English/Latin": [
        "lit", "fire", "cap", "sus", "vibe", "mood", "slay", "bussin",
        "lowkey", "highkey", "fr", "ngl", "tbh", "af", "lol", "lmao",
        "bruh", "bro", "dude", "yo", "yeah", "yep", "nah", "gonna", "wanna"
    ],
    "Spanish": [
        "chido", "chingón", "guay", "tío", "mola", "flipar", "pibe",
        "boludo", "güey", "wey", "chévere", "bacán", "joder"
    ],
    "French": [
        "ouf", "grave", "chelou", "meuf", "mec", "kiffer", "trop",
        "genre", "stylé", "canon", "mortel"
    ],
    "German": [
        "geil", "krass", "cool", "abgefahren", "Alter", "Typ",
        "checken", "chillen", "funky"
    ],
    "Italian": [
        "figo", "fichissimo", "ganzo", "sgravato", "fratello", "bello"
    ],
    "Portuguese": [
        "maneiro", "legal", "cara", "mano", "meu", "massa", "da hora"
    ]
}

def check_slang_semantics(tokens_by_language: Dict[str, List[Tuple[str, float]]]) -> Dict[str, int]:
    """
    Check how many tokens match known slang patterns.
    """
    slang_matches = defaultdict(int)

    for lang, tokens in tokens_by_language.items():
        if lang in SLANG_INDICATORS:
            for token, _ in tokens:
                token_lower = token.lower().strip()
                for slang_word in SLANG_INDICATORS[lang]:
                    if slang_word in token_lower:
                        slang_matches[lang] += 1
                        break

    return slang_matches


print("\n" + "="*80)
print("SEMANTIC SLANG ANALYSIS ACROSS LANGUAGES")
print("="*80)

for result in vocabproj_results:
    print(f"\n{'─'*80}")
    print(f"Feature {result.feature_id} ({result.feature_category})")
    print(f"{'─'*80}")

    lang_dist = crosslingual_analysis[result.feature_id]
    slang_matches = check_slang_semantics(lang_dist)

    if slang_matches:
        print(f"\n✓ Found slang indicators in:")
        for lang, count in slang_matches.items():
            total = len(lang_dist[lang])
            percentage = (count / total * 100) if total > 0 else 0
            print(f"  {lang:20s}: {count}/{total} tokens ({percentage:.1f}%)")
    else:
        print("\n⚠ No direct slang matches found in known patterns")

    # Additional analysis: look for informal markers
    informal_markers = {
        'contractions': 0,
        'abbreviations': 0,
        'exclamations': 0,
        'intensifiers': 0
    }

    for token, _ in result.top_promoted_tokens[:50]:
        token_lower = token.lower().strip()

        # Contractions (English)
        if "'" in token_lower and any(x in token_lower for x in ["n't", "'ll", "'ve", "'re", "'m", "'d"]):
            informal_markers['contractions'] += 1

        # Short forms/abbreviations
        if len(token_lower) <= 3 and token_lower.isalpha():
            informal_markers['abbreviations'] += 1

        # Exclamations
        if '!' in token or '?' in token:
            informal_markers['exclamations'] += 1

        # Common intensifiers
        intensifiers = ['very', 'so', 'really', 'super', 'totally', 'completely',
                       'absolutely', 'extremely', 'too', 'quite']
        if any(word in token_lower for word in intensifiers):
            informal_markers['intensifiers'] += 1

    print(f"\nInformal Language Markers:")
    for marker, count in informal_markers.items():
        if count > 0:
            print(f"  {marker.capitalize():20s}: {count}")


SEMANTIC SLANG ANALYSIS ACROSS LANGUAGES

────────────────────────────────────────────────────────────────────────────────
Feature 35440 (universal)
────────────────────────────────────────────────────────────────────────────────

✓ Found slang indicators in:
  English/Latin       : 7/47 tokens (14.9%)

Informal Language Markers:
  Abbreviations       : 2
  Exclamations        : 1

────────────────────────────────────────────────────────────────────────────────
Feature 33236 (universal)
────────────────────────────────────────────────────────────────────────────────

✓ Found slang indicators in:
  English/Latin       : 2/39 tokens (5.1%)

Informal Language Markers:
  Intensifiers        : 2

────────────────────────────────────────────────────────────────────────────────
Feature 93521 (universal)
────────────────────────────────────────────────────────────────────────────────

✓ Found slang indicators in:
  English/Latin       : 5/46 tokens (10.9%)

Informal Language Markers:

───────

## 11. Visualize Cross-Lingual Distribution

In [16]:
def plot_language_distribution():
    """
    Visualize the distribution of languages in promoted tokens across features.
    """
    plot_data = []

    for result in vocabproj_results:
        lang_dist = crosslingual_analysis[result.feature_id]
        for lang, tokens in lang_dist.items():
            plot_data.append({
                'Feature': f"{result.feature_id} ({result.feature_category})",
                'Language': lang,
                'Count': len(tokens),
                'Category': result.feature_category
            })

    df = pd.DataFrame(plot_data)

    # Create stacked bar chart
    fig = px.bar(
        df,
        x='Feature',
        y='Count',
        color='Language',
        title='Language Distribution in Top 50 Promoted Tokens by Feature',
        labels={'Count': 'Number of Tokens'},
        height=600
    )

    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

    # Also create a summary by category
    category_summary = df.groupby(['Category', 'Language'])['Count'].sum().reset_index()

    fig2 = px.bar(
        category_summary,
        x='Category',
        y='Count',
        color='Language',
        title='Language Distribution by Feature Category',
        labels={'Count': 'Total Token Count'},
        height=500
    )

    fig2.show()

plot_language_distribution()

## 12. Manual Inspection: Cross-Lingual Slang Examples

Let's examine specific non-English tokens to manually verify if they preserve slang semantics.

In [17]:
def display_crosslingual_examples():
    """
    Display interesting cross-lingual tokens for manual inspection.
    """
    print("\n" + "="*80)
    print("CROSS-LINGUAL SLANG TOKEN EXAMPLES")
    print("="*80)
    print("\nThese are non-English tokens promoted by slang features.")
    print("Manual inspection can reveal if they carry slang/informal semantics.\n")

    for result in vocabproj_results:
        print(f"\n{'─'*80}")
        print(f"Feature {result.feature_id} ({result.feature_category})")
        print(f"{'─'*80}")

        lang_dist = crosslingual_analysis[result.feature_id]

        # Focus on non-English languages
        interesting_langs = [lang for lang in lang_dist.keys()
                           if lang not in ['English/Latin', 'Other/Symbol', 'empty']]

        if interesting_langs:
            for lang in interesting_langs:
                tokens = lang_dist[lang][:10]  # Top 10 per language
                if tokens:
                    print(f"\n  {lang}:")
                    for token, weight in tokens:
                        print(f"    {token:25s} (weight: {weight:7.4f})")
        else:
            print("\n  No significant non-English tokens found.")

display_crosslingual_examples()


CROSS-LINGUAL SLANG TOKEN EXAMPLES

These are non-English tokens promoted by slang features.
Manual inspection can reveal if they carry slang/informal semantics.


────────────────────────────────────────────────────────────────────────────────
Feature 35440 (universal)
────────────────────────────────────────────────────────────────────────────────

  Arabic:
     الرياضيه                 (weight: 38.0000)

────────────────────────────────────────────────────────────────────────────────
Feature 33236 (universal)
────────────────────────────────────────────────────────────────────────────────

  Devanagari/Hindi:
     ब्रेकडाउन                (weight: 32.5000)

  French:
    évaluateur                (weight: 31.8750)
     fédé                     (weight: 30.7500)
     informée                 (weight: 29.0000)

  Cyrillic/Russian:
    цездатний                 (weight: 31.0000)
     Савезне                  (weight: 30.5000)
    хьтан                     (weight: 28.8750)
     Мекси

## 13. Test Ambiguity Preservation

Test if features preserve the **ambiguous/context-dependent** nature of slang across languages.

In [18]:
# Test sentences in different languages with ambiguous slang
multilingual_slang_examples = {
    "English": [
        "That's so fire!",  # slang: excellent
        "This is fire.",  # could be literal or slang
        "No cap, that's amazing",  # slang: no lie
        "Put a cap on it",  # literal: cover it
    ],
    "Spanish": [
        "Eso está muy guay!",  # slang: that's cool (Spain)
        "Qué chido!",  # slang: how cool (Mexico)
        "Está muy padre",  # slang: it's very cool (Mexico)
    ],
    "French": [
        "C'est trop ouf!",  # verlan slang: that's crazy
        "C'est grave stylé",  # slang: that's really stylish
        "C'est un fou",  # literal/semi-slang: he's crazy
    ],
    "German": [
        "Das ist echt geil!",  # slang: that's awesome
        "Voll krass!",  # slang: totally awesome/crazy
    ]
}

def test_multilingual_activation(feature_id: int, category: str):
    """
    Test how features activate on slang in different languages.
    """
    print(f"\n{'='*80}")
    print(f"Testing Feature {feature_id} ({category}) on Multilingual Slang")
    print(f"{'='*80}\n")

    all_activations = {}

    for language, examples in multilingual_slang_examples.items():
        print(f"\n{language}:")
        print(f"{'-'*60}")

        activations = []
        for example in examples:
            tokens = model.to_tokens(example, prepend_bos=True)

            # Get activations
            _, cache = model.run_with_cache(tokens)
            acts = cache[sae.cfg.hook_name]
            sae_acts = sae.encode(acts)

            # Max activation for this feature
            feature_act = sae_acts[0, :, feature_id].max().item()
            activations.append(feature_act)

            print(f"  {example:40s} | Act: {feature_act:6.3f}")

        all_activations[language] = activations
        mean_act = np.mean(activations)
        print(f"  Mean activation: {mean_act:.3f}")

    return all_activations


print("\n" + "#"*80)
print("# MULTILINGUAL SLANG ACTIVATION TEST")
print("#"*80)

multilingual_results = {}

for category, feature_ids in SLANG_FEATURES.items():
    for feature_id in feature_ids:
        result = test_multilingual_activation(feature_id, category)
        multilingual_results[feature_id] = result

# Summary comparison
print("\n\n" + "="*80)
print("CROSS-LINGUAL ACTIVATION SUMMARY")
print("="*80)

for category, feature_ids in SLANG_FEATURES.items():
    for feature_id in feature_ids:
        print(f"\nFeature {feature_id} ({category}):")
        result = multilingual_results[feature_id]

        lang_means = {lang: np.mean(acts) for lang, acts in result.items()}
        sorted_langs = sorted(lang_means.items(), key=lambda x: x[1], reverse=True)

        for lang, mean_act in sorted_langs:
            print(f"  {lang:15s}: {mean_act:.3f}")

        # Check if activations are consistent across languages
        activations = list(lang_means.values())
        std = np.std(activations)
        mean = np.mean(activations)
        cv = (std / mean) if mean > 0 else 0  # coefficient of variation

        print(f"\n  Consistency (lower CV = more consistent): {cv:.2f}")
        if cv < 0.5:
            print(f"  ✓ Feature shows consistent activation across languages")
        else:
            print(f"  ⚠ Feature shows variable activation across languages")


################################################################################
# MULTILINGUAL SLANG ACTIVATION TEST
################################################################################

Testing Feature 35440 (universal) on Multilingual Slang


English:
------------------------------------------------------------


NameError: name 'model' is not defined

## 15. Export Enhanced Results

In [ ]:
def export_complete_analysis():
    """
    Export all analysis results including cross-lingual findings.
    """
    # Combine all results
    complete_results = []

    for vp_result in vocabproj_results:
        feature_id = vp_result.feature_id

        # Get corresponding TokenChange results
        tc_results = [r for r in all_results if r.feature_id == feature_id]

        # Get cross-lingual analysis
        lang_dist = crosslingual_analysis[feature_id]

        # Get multilingual activation results
        multi_acts = multilingual_results.get(feature_id, {})

        result_entry = {
            'feature_id': feature_id,
            'category': vp_result.feature_category,

            # VocabProj results
            'vp_top_tokens': [t[0] for t in vp_result.top_promoted_tokens[:20]],
            'vp_mean_weight': vp_result.mean_absolute_weight,

            # TokenChange results (highest amplification)
            'tc_results': [{
                'amplification': r.amplification,
                'top_tokens': [t[0] for t in r.top_increased_tokens[:20]],
                'mean_change': r.mean_absolute_change
            } for r in tc_results],

            # Cross-lingual analysis
            'languages_detected': {lang: len(tokens) for lang, tokens in lang_dist.items()},
            'non_english_tokens': {
                lang: [t[0] for t in tokens[:10]]
                for lang, tokens in lang_dist.items()
                if lang not in ['English/Latin', 'empty', 'Other/Symbol']
            },

            # Multilingual activation
            'multilingual_activations': {
                lang: float(np.mean(acts))
                for lang, acts in multi_acts.items()
            }
        }

        complete_results.append(result_entry)

    # Export to JSON
    with open('complete_slang_analysis.json', 'w', encoding='utf-8') as f:
        json.dump(complete_results, f, indent=2, ensure_ascii=False)

    print("✓ Complete analysis exported to complete_slang_analysis.json")

    # Also create a summary report
    report_lines = []
    report_lines.append("="*80)
    report_lines.append("SLANG FEATURE ANALYSIS SUMMARY REPORT")
    report_lines.append("="*80)
    report_lines.append("")

    for entry in complete_results:
        report_lines.append(f"\nFeature {entry['feature_id']} ({entry['category']})")
        report_lines.append("-" * 60)

        # VocabProj summary
        report_lines.append("\nVocabProj Top 10 Tokens:")
        report_lines.append("  " + ", ".join(entry['vp_top_tokens'][:10]))

        # Cross-lingual summary
        report_lines.append("\nLanguage Distribution:")
        for lang, count in entry['languages_detected'].items():
            report_lines.append(f"  {lang}: {count} tokens")

        # Multilingual activation
        if entry['multilingual_activations']:
            report_lines.append("\nMultilingual Activation:")
            for lang, act in entry['multilingual_activations'].items():
                report_lines.append(f"  {lang}: {act:.3f}")

        report_lines.append("")

    # Write report
    with open('slang_analysis_report.txt', 'w', encoding='utf-8') as f:
        f.write("\n".join(report_lines))

    print("✓ Summary report exported to slang_analysis_report.txt")

    return complete_results

complete_results = export_complete_analysis()
print("\n✓ All exports completed successfully!")

#run VocabProj on the AVG of the vectors

In [19]:
# Define feature combinations to average
FEATURE_COMBINATIONS = {
    "universal_avg": {
        "features": [35440, 93521],  # Only the true slang features
        "category": "universal_combined",
        "description": "Average of true universal slang features (35440 + 93521)"
    },
    "all_universal_avg": {
        "features": [35440, 33236, 93521],  # All three universal features
        "category": "universal_all",
        "description": "Average of all three universal features"
    },
    "code_vs_natural": {
        "features": [51811, 90871],  # The opposite pair
        "category": "code_natural_opposition",
        "description": "Average of code/markup vs natural language features"
    },
    "true_slang_all": {
        "features": [35440, 93521, 90871],  # True slang + literalness (inverse)
        "category": "slang_with_literalness",
        "description": "True slang features combined with literalness"
    }
}


def vocabproj_averaged_features(
    feature_ids: List[int],
    combination_name: str,
    category: str,
    description: str,
    top_k: int = 50
) -> VocabProjResult:
    """
    Analyze averaged feature vectors using VocabProj method.

    Args:
        feature_ids: List of SAE feature indices to average
        combination_name: Name for this combination
        category: Category label
        description: Description of what this combination represents
        top_k: Number of top tokens to return

    Returns:
        VocabProjResult with analysis results
    """
    print(f"\n{'='*80}")
    print(f"Analyzing AVERAGED features: {combination_name}")
    print(f"Features: {feature_ids}")
    print(f"Description: {description}")
    print(f"{'='*80}")

    # 1. Get feature vectors and average them
    feature_vectors = []
    for fid in feature_ids:
        feature_vector = sae.W_dec[fid, :]  # Shape: [d_model]
        feature_vectors.append(feature_vector)

    # Average the feature vectors
    averaged_vector = torch.stack(feature_vectors).mean(dim=0)  # Shape: [d_model]
    print(f"Averaged {len(feature_ids)} feature vectors")

    # 2. Apply LayerNorm (model's final layer norm)
    ln = tl_model.ln_final
    normalized_feature = ln(averaged_vector.unsqueeze(0)).squeeze(0)

    # 3. Project to vocabulary space using unembedding matrix
    W_U = tl_model.W_U  # Shape: [d_model, vocab_size]
    vocab_logits = normalized_feature @ W_U  # Shape: [vocab_size]

    # 4. Convert to float32 for numpy compatibility
    vocab_logits_np = vocab_logits.float().cpu().numpy()

    # Get indices of top and bottom tokens
    top_indices = np.argsort(vocab_logits_np)[-top_k:][::-1]
    bottom_indices = np.argsort(vocab_logits_np)[:top_k]

    top_promoted = [
        (tl_model.to_string(int(idx)), float(vocab_logits_np[idx]))
        for idx in top_indices
    ]

    top_suppressed = [
        (tl_model.to_string(int(idx)), float(vocab_logits_np[idx]))
        for idx in bottom_indices
    ]

    # 5. Calculate statistics
    mean_abs_weight = float(np.mean(np.abs(vocab_logits_np)))
    max_weight = float(np.max(np.abs(vocab_logits_np)))

    print(f"\nResults:")
    print(f"  Mean absolute weight: {mean_abs_weight:.4f}")
    print(f"  Max weight: {max_weight:.4f}")

    # Display top tokens
    print(f"\n  Top 10 promoted tokens:")
    for i, (token, weight) in enumerate(top_promoted[:10], 1):
        print(f"    {i:2d}. {token:20s} {weight:8.4f}")

    return VocabProjResult(
        feature_id=f"avg_{'-'.join(map(str, feature_ids))}",  # Combined ID as string
        feature_category=category,
        top_promoted_tokens=top_promoted,
        top_suppressed_tokens=top_suppressed,
        mean_absolute_weight=mean_abs_weight,
        max_weight=max_weight
    )


# Run VocabProj on averaged feature combinations
print("\n" + "#"*80)
print("# VOCABPROJ ANALYSIS ON AVERAGED FEATURE VECTORS")
print("#"*80)

averaged_results = []

for combo_name, combo_info in FEATURE_COMBINATIONS.items():
    result = vocabproj_averaged_features(
        feature_ids=combo_info["features"],
        combination_name=combo_name,
        category=combo_info["category"],
        description=combo_info["description"],
        top_k=50
    )
    averaged_results.append(result)

print(f"\n\nCompleted VocabProj analysis on {len(averaged_results)} averaged feature combinations")


################################################################################
# VOCABPROJ ANALYSIS ON AVERAGED FEATURE VECTORS
################################################################################

Analyzing AVERAGED features: universal_avg
Features: [35440, 93521]
Description: Average of true universal slang features (35440 + 93521)
Averaged 2 feature vectors

Results:
  Mean absolute weight: 7.5477
  Max weight: 48.5000

  Top 10 promoted tokens:
     1.  dudes                48.5000
     2.  vibes                46.2500
     3.  badass               45.5000
     4.  funky                44.7500
     5.  guys                 43.7500
     6.  vibe                 43.5000
     7.  dude                 42.7500
     8.  swag                 42.5000
     9.  streetwear           39.7500
    10.  nakalista            39.7500

Analyzing AVERAGED features: all_universal_avg
Features: [35440, 33236, 93521]
Description: Average of all three universal features
Averaged 3 feature 

In [20]:
# Compare individual features vs their averages
print("\n" + "="*80)
print("COMPARISON: Individual Features vs Averaged Combinations")
print("="*80)

# Compare true slang features individually vs averaged
print("\n" + "-"*80)
print("TRUE SLANG FEATURES: Individual vs Average")
print("-"*80)

print("\nFeature 35440 (individual) top 5:")
f35440 = [r for r in vocabproj_results if r.feature_id == 35440][0]
for i, (token, weight) in enumerate(f35440.top_promoted_tokens[:5], 1):
    print(f"  {i}. {token:20s} {weight:.2f}")

print("\nFeature 93521 (individual) top 5:")
f93521 = [r for r in vocabproj_results if r.feature_id == 93521][0]
for i, (token, weight) in enumerate(f93521.top_promoted_tokens[:5], 1):
    print(f"  {i}. {token:20s} {weight:.2f}")

print("\nAveraged (35440 + 93521) top 5:")
avg_true_slang = [r for r in averaged_results if "35440" in str(r.feature_id) and "93521" in str(r.feature_id)][0]
for i, (token, weight) in enumerate(avg_true_slang.top_promoted_tokens[:5], 1):
    print(f"  {i}. {token:20s} {weight:.2f}")

print("\n" + "-"*80)
print("ANALYSIS:")
print("-"*80)

# Get all unique tokens from individual features
individual_tokens = set([t[0] for t in f35440.top_promoted_tokens[:20]] +
                       [t[0] for t in f93521.top_promoted_tokens[:20]])
averaged_tokens = set([t[0] for t in avg_true_slang.top_promoted_tokens[:20]])

# Find overlap and unique tokens
overlap = individual_tokens & averaged_tokens
only_in_individual = individual_tokens - averaged_tokens
only_in_averaged = averaged_tokens - individual_tokens

print(f"\nTop 20 token overlap: {len(overlap)}/40 possible")
print(f"Tokens only in individual features: {len(only_in_individual)}")
print(f"New tokens emerging from averaging: {len(only_in_averaged)}")

if only_in_averaged:
    print(f"\nNew tokens from averaging (not in individual top 20s):")
    for token in list(only_in_averaged)[:10]:
        print(f"  - {token}")


COMPARISON: Individual Features vs Averaged Combinations

--------------------------------------------------------------------------------
TRUE SLANG FEATURES: Individual vs Average
--------------------------------------------------------------------------------

Feature 35440 (individual) top 5:
  1.  dudes               47.50
  2. hoeddwyd             45.75
  3.  guys                43.75
  4.  funky               43.00
  5.  fellas              42.50

Feature 93521 (individual) top 5:
  1.  emojis              38.25
  2.  vibes               38.00
  3.  swag                36.25
  4.  חיצוניים            36.00
  5.  kaarangay           35.25

Averaged (35440 + 93521) top 5:
  1.  dudes               48.50
  2.  vibes               46.25
  3.  badass              45.50
  4.  funky               44.75
  5.  guys                43.75

--------------------------------------------------------------------------------
ANALYSIS:
-------------------------------------------------------------